In [ ]:
!pip install tabulate

In [ ]:
! pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 32.0 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import shutil

source_path = '/content/drive/MyDrive/SPORT_METADATA/Structure data/Gym_exercises/scrap'

destination_path = '/content/Gym_exercises/'

shutil.copytree(source_path, destination_path)

print(f"Directory copied from {source_path} to {destination_path}")

Directory copied from /content/drive/MyDrive/SPORT_METADATA/Structure data/Gym_exercises/scrap to /content/Gym_exercises/


In [ ]:
source_file = '/content/drive/MyDrive/SPORT_METADATA/Structure data/Gym_exercises/exercises_final_verified.csv'

destination_path = '/content/Gym_exercises/'

shutil.copy2(source_file, destination_path)

print(f"Directory copied from {source_file} to {destination_path}")

Directory copied from /content/drive/MyDrive/SPORT_METADATA/Structure data/Gym_exercises/exercises_final_verified.csv to /content/Gym_exercises/


# peek on the api source data

In [ ]:
import pandas as pd
from tabulate import tabulate

df_exercise = pd.read_csv("/content/Gym_exercises/exercises_final_verified.csv")
df_exercise.drop(columns="instructions",inplace=True)

print("The column names of the API data:")
print(df_exercise.columns.tolist())
print("\n" + "="*60)
df_exercise.shape

The column names of the API data:
['exercise_name', 'primary_muscles', 'secondary_muscles', 'equipment', 'level', 'force', 'category', 'mechanic', 'text_chunk']



(873, 9)

# peek on scrap source data

In [ ]:
import pandas as pd
from tabulate import tabulate

# Corrected code - using read_json for JSON file
df_exercise = pd.read_json("/content/Gym_exercises/final_exrx_database.json")

# Check if 'instructions' column exists before dropping
if 'instructions' in df_exercise.columns:
    df_exercise.drop(columns="instructions", inplace=True)
else:
    print("Note: 'instructions' column not found in the dataset")

print("The column names of the API data:")
print(df_exercise.columns.tolist())
print("\n" + "="*60)

# Additional useful information:
print("\nDataset Info:")
print(f"Total rows: {len(df_exercise)}")
print(f"Total columns: {len(df_exercise.columns)}")
print("\n" + "="*60)

print("\nFirst 5 rows:")
print(tabulate(df_exercise.head(), headers='keys', tablefmt='grid'))
print("\n" + "="*60)


The column names of the API data:
['breadcrumb_path', 'parent_category', 'source_url', 'exercise_name', 'utility', 'mechanics', 'force', 'comments', 'muscles', 'easier', 'harder', 'function', 'intensity', 'force4']


Dataset Info:
Total rows: 873
Total columns: 14


First 5 rows:
+----+--------------------------------------------------------------------------+-------------------+----------------------------------------------------------------------------+-------------------------------------------+--------------------+-------------+---------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------

# normalize the api and scraped data

In [ ]:
import pandas as pd
import ast
import numpy as np


df_scrap=pd.read_json("/content/Gym_exercises/final_exrx_database.json")
df_api=pd.read_csv("/content/Gym_exercises/exercises_final_verified.csv")

# ---------------------------------------------------------
# 1. Preprocess Scraped Data
# ---------------------------------------------------------
df_scrap_normalized = pd.DataFrame()

# Standardize the exercise name for exact matching later
df_scrap_normalized['exercise_name'] = df_scrap['exercise_name'].str.lower().str.strip()
df_scrap_normalized['category'] = df_scrap['parent_category']
df_scrap_normalized['mechanic'] = df_scrap['mechanics']
df_scrap_normalized['force'] = df_scrap['force']
df_scrap_normalized['source_url'] = df_scrap['source_url']
df_scrap_normalized['source'] = 'exrx_scraper'

# Safely extract muscles from the dictionary string
def extract_muscles(muscle_val, key):
    if pd.isna(muscle_val):
        return None
    try:
        muscle_dict = ast.literal_eval(muscle_val)
        if key == 'primary':
            return ', '.join(muscle_dict.get('target', []))
        elif key == 'secondary':
            synergists = muscle_dict.get('synergists', [])
            stabilizers = muscle_dict.get('stabilizers', [])
            secondary = [m for m in (synergists + stabilizers) if m != 'None']
            return ', '.join(secondary) if secondary else None
    except:
        return None

df_scrap_normalized['primary_muscles'] = df_scrap['muscles'].apply(lambda x: extract_muscles(x, 'primary'))
df_scrap_normalized['secondary_muscles'] = df_scrap['muscles'].apply(lambda x: extract_muscles(x, 'secondary'))

# Set columns missing in scraped data to None
df_scrap_normalized['equipment'] = None
df_scrap_normalized['level'] = None

# Build the rich text chunk for RAG
def build_scrap_text_chunk(row):
    parts = []
    if pd.notna(row.get('utility')): parts.append(f"Utility: {row['utility']}.")
    if pd.notna(row.get('comments')): parts.append(f"Comments: {row['comments']}")
    if pd.notna(row.get('easier')): parts.append(f"To make it easier: {row['easier']}.")
    if pd.notna(row.get('harder')): parts.append(f"To make it harder: {row['harder']}.")
    return " ".join(parts) if parts else None

df_scrap_normalized['text_chunk'] = df_scrap.apply(build_scrap_text_chunk, axis=1)

# ---------------------------------------------------------
# 2. Preprocess API Data
# ---------------------------------------------------------
df_api_normalized = pd.DataFrame()

# Standardize the exercise name
df_api_normalized['exercise_name'] = df_api['exercise_name'].str.lower().str.strip()
df_api_normalized['category'] = df_api['category']
df_api_normalized['primary_muscles'] = df_api['primary_muscles']
df_api_normalized['secondary_muscles'] = df_api['secondary_muscles']
df_api_normalized['equipment'] = df_api['equipment']
df_api_normalized['mechanic'] = df_api['mechanic']
df_api_normalized['force'] = df_api['force']
df_api_normalized['level'] = df_api['level']
df_api_normalized['text_chunk'] = df_api['text_chunk']
df_api_normalized['source_url'] = None
df_api_normalized['source'] = 'api'

# ---------------------------------------------------------
# 3. Concatenate Both Datasets
# ---------------------------------------------------------
# Ensure column order matches exactly
final_columns = [
    'exercise_name', 'category', 'primary_muscles', 'secondary_muscles',
    'equipment', 'mechanic', 'force', 'level', 'source', 'source_url', 'text_chunk'
]

df_combined_raw = pd.concat([
    df_scrap_normalized[final_columns],
    df_api_normalized[final_columns]
], ignore_index=True)

# ---------------------------------------------------------
# 4. Group and Merge Duplicates
# ---------------------------------------------------------
# Helper function: Gets the first non-null value for metadata so we don't lose info
def first_non_null(series):
    valid_data = series.dropna()
    return valid_data.iloc[0] if not valid_data.empty else None

# Helper function: Combines text chunks cleanly
def combine_text(series):
    valid_data = series.dropna().astype(str)
    # Use .unique() to avoid duplicating the exact same text if it happens to exist
    return '\n\n--- Additional Info ---\n\n'.join(valid_data.unique()) if not valid_data.empty else None

# Define how each column should behave when merging two rows with the same name
aggregation_rules = {
    'category': first_non_null,
    'primary_muscles': first_non_null,
    'secondary_muscles': first_non_null,
    'equipment': first_non_null,
    'mechanic': first_non_null,
    'force': first_non_null,
    'level': first_non_null,
    'source_url': first_non_null,
    'source': lambda x: ' + '.join(x.dropna().unique()), # Tags it as "api + exrx_scraper"
    'text_chunk': combine_text
}


# addational pahse to merge same exercise name (almost same exercise usong fuzzy)

In [ ]:
import pandas as pd
from rapidfuzz import process, fuzz

# --- Assume df_scrap_normalized and df_api_normalized are already created ---
# (Using the preprocessing code from the previous step)

api_exercise_names = df_api_normalized['exercise_name'].tolist()

# 1. Define the fuzzy matching function
def find_best_match(scraped_name, choices, threshold=85):
    """
    Finds the best match for a scraped name within the API names.
    If the match score is above the threshold, return the API name.
    Otherwise, keep the original scraped name.
    """
    if pd.isna(scraped_name):
        return scraped_name

    # extractOne returns a tuple: (best_match_string, score, index)
    match = process.extractOne(
        scraped_name,
        choices,
        scorer=fuzz.token_sort_ratio
    )

    if match:
        best_string, score, _ = match
        if score >= threshold:
            return best_string # Return the matched API name

    return scraped_name # Keep original if no good match found

# 2. Apply fuzzy matching to the scraped data
print("Fuzzy matching exercise names...")
df_scrap_normalized['unified_exercise_name'] = df_scrap_normalized['exercise_name'].apply(
    lambda x: find_best_match(x, api_exercise_names, threshold=85)
)

# Also copy the API names to the same new column so they align perfectly
df_api_normalized['unified_exercise_name'] = df_api_normalized['exercise_name']

# 3. Concatenate Both Datasets
final_columns = [
    'unified_exercise_name', 'category', 'primary_muscles', 'secondary_muscles',
    'equipment', 'mechanic', 'force', 'level', 'source', 'source_url', 'text_chunk'
]

df_combined_raw = pd.concat([
    df_scrap_normalized[final_columns],
    df_api_normalized[final_columns]
], ignore_index=True)

# 4. Group and Merge using the newly unified name
def first_non_null(series):
    valid_data = series.dropna()
    return valid_data.iloc[0] if not valid_data.empty else None

def combine_text(series):
    valid_data = series.dropna().astype(str)
    return '\n\n--- Additional Info ---\n\n'.join(valid_data.unique()) if not valid_data.empty else None

aggregation_rules = {
    'category': first_non_null,
    'primary_muscles': first_non_null,
    'secondary_muscles': first_non_null,
    'equipment': first_non_null,
    'mechanic': first_non_null,
    'force': first_non_null,
    'level': first_non_null,
    'source_url': first_non_null,
    'source': lambda x: ' + '.join(x.dropna().unique()),
    'text_chunk': combine_text
}

# Group by 'unified_exercise_name' instead of the raw 'exercise_name'
df_combined = df_combined_raw.groupby('unified_exercise_name', as_index=False).agg(aggregation_rules)

# Rename the column back to 'exercise_name' for cleanliness
df_combined.rename(columns={'unified_exercise_name': 'exercise_name'}, inplace=True)

print(f"Final dataset ready with {len(df_combined)} unique exercises.")

Fuzzy matching exercise names...
Final dataset ready with 1506 unique exercises.


In [ ]:
# 1. CRITICAL: Synthesize a fallback text chunk for the 7 missing rows
missing_text_mask = df_combined['text_chunk'].isna()
df_combined.loc[missing_text_mask, 'text_chunk'] = df_combined[missing_text_mask].apply(
    lambda row: f"Exercise: {row['exercise_name']}. This is a fitness movement categorized under {row['category']}.",
    axis=1
)

# 2. METADATA SAFETY: Fill missing categorical data for the vector database
metadata_cols = [
    'primary_muscles', 'secondary_muscles',
    'level', 'equipment', 'force', 'mechanic'
]
df_combined[metadata_cols] = df_combined[metadata_cols].fillna("Not Specified")

# 3. URL SAFETY: Fill missing URLs so front-end links don't break
df_combined['source_url'] = df_combined['source_url'].fillna("No URL provided")

# Check to confirm it's clean
print("Missing text chunks:", df_combined['text_chunk'].isna().sum()) # Should be 0

Missing text chunks: 0


# summary on merged data

In [ ]:

def display_data_quality_report(df, df_name="df_combined"):
    """
    Display comprehensive information about missing data and duplicates in a dataframe
    """
    print("="*80)
    print(f"DATA QUALITY REPORT FOR: {df_name}")
    print("="*80)

    # 1. BASIC INFO ABOUT MISSING DATA
    print("\n" + "="*40)
    print("1. MISSING DATA SUMMARY")
    print("="*40)

    # Total missing values
    total_missing = df.isnull().sum().sum()
    total_cells = df.size
    missing_percentage = (total_missing / total_cells) * 100

    print(f"\nTotal missing values: {total_missing:,} out of {total_cells:,} cells ({missing_percentage:.2f}%)")
    print(f"Rows with at least one missing value: {df.isnull().any(axis=1).sum():,} out of {len(df):,} rows")

    # Missing values by column
    missing_by_column = pd.DataFrame({
        'Missing Values': df.isnull().sum(),
        'Missing %': (df.isnull().sum() / len(df) * 100).round(2),
        'Data Type': df.dtypes
    }).sort_values('Missing Values', ascending=False)

    print("\nMissing values by column:")
    print(missing_by_column[missing_by_column['Missing Values'] > 0])

    # Columns with no missing data
    cols_complete = df.columns[df.isnull().sum() == 0].tolist()
    if cols_complete:
        print(f"\nColumns with NO missing data ({len(cols_complete)} columns):")
        print(f"  {', '.join(cols_complete[:10])}{'...' if len(cols_complete) > 10 else ''}")

    # 2. DETAILED MISSING DATA PATTERNS
    print("\n" + "="*40)
    print("2. MISSING DATA PATTERNS")
    print("="*40)

    # Missing value patterns
    if total_missing > 0:
        # Create missing indicator matrix
        missing_pattern = df.isnull().sum(axis=1).value_counts().sort_index()
        print("\nRows with number of missing values:")
        for n_missing, count in missing_pattern.items():
            percentage = (count / len(df)) * 100
            print(f"  {n_missing} missing values: {count:,} rows ({percentage:.2f}%)")

        # Most common missing patterns
        print("\nMost common missing value patterns (top 5):")
        missing_patterns = df.isnull().astype(int).groupby(df.columns.tolist()).size().sort_values(ascending=False).head()
        for pattern, count in missing_patterns.items():
            print(f"  Pattern {pattern}: {count} rows")
    else:
        print("\nNo missing data patterns to display - dataframe has no missing values!")

    # 3. DUPLICATES INFORMATION
    print("\n" + "="*40)
    print("3. DUPLICATES SUMMARY")
    print("="*40)

    # Check for duplicates
    duplicate_rows = df.duplicated().sum()
    duplicate_percentage = (duplicate_rows / len(df)) * 100 if len(df) > 0 else 0

    print(f"\nTotal duplicate rows: {duplicate_rows:,} out of {len(df):,} rows ({duplicate_percentage:.2f}%)")

    if duplicate_rows > 0:
        # Show duplicate counts
        duplicate_counts = df[df.duplicated(keep=False)].groupby(df.columns.tolist()).size().sort_values(ascending=False)
        print(f"\nNumber of unique duplicate groups: {len(duplicate_counts)}")
        print(f"Average duplicates per group: {duplicate_counts.mean():.2f}")
        print(f"Max duplicates in a single group: {duplicate_counts.max()}")

        # Show sample of duplicates
        print("\nSample of duplicate groups (first 3):")
        duplicate_samples = duplicate_counts.head(3)
        for i, (index, count) in enumerate(duplicate_samples.items()):
            print(f"\n  Group {i+1} (appears {count} times):")
            # Get one row from this duplicate group
            if isinstance(index, tuple):
                # Find a sample row matching this pattern
                mask = pd.Series([True] * len(df))
                for col, val in zip(df.columns, index):
                    mask &= (df[col] == val)
                sample_row = df[mask].iloc[0]
                print(f"    {sample_row.to_dict()}")
    else:
        print("\nNo duplicate rows found!")

    # 4. COLUMN-BY-COLUMN DETAILED ANALYSIS
    print("\n" + "="*40)
    print("4. COLUMN-BY-COLUMN DETAILED ANALYSIS")
    print("="*40)

    for col in df.columns:
        missing_count = df[col].isnull().sum()
        missing_pct = (missing_count / len(df)) * 100
        dtype = df[col].dtype
        unique_count = df[col].nunique()

        print(f"\n{col}:")
        print(f"  Type: {dtype}")
        print(f"  Missing: {missing_count:,} ({missing_pct:.2f}%)")
        print(f"  Unique values: {unique_count:,}")

        if missing_count > 0:
            # For columns with missing data, show some context
            if pd.api.types.is_numeric_dtype(df[col]):
                print(f"  Mean (excluding missing): {df[col].mean():.2f}")
                print(f"  Median (excluding missing): {df[col].median():.2f}")
            else:
                # For categorical/text, show most common non-null values
                top_values = df[col].value_counts().head(3)
                if not top_values.empty:
                    print(f"  Top values:")
                    for val, count in top_values.items():
                        pct = (count / len(df[col].dropna())) * 100
                        print(f"    {val}: {count} ({pct:.1f}% of non-null)")



    print("\n" + "="*80)
    print("END OF DATA QUALITY REPORT")
    print("="*80)

# Run the comprehensive report
display_data_quality_report(df_combined)

DATA QUALITY REPORT FOR: df_combined

1. MISSING DATA SUMMARY

Total missing values: 0 out of 16,566 cells (0.00%)
Rows with at least one missing value: 0 out of 1,506 rows

Missing values by column:
Empty DataFrame
Columns: [Missing Values, Missing %, Data Type]
Index: []

Columns with NO missing data (11 columns):
  exercise_name, category, primary_muscles, secondary_muscles, equipment, mechanic, force, level, source_url, source...

2. MISSING DATA PATTERNS

No missing data patterns to display - dataframe has no missing values!

3. DUPLICATES SUMMARY

Total duplicate rows: 0 out of 1,506 rows (0.00%)

No duplicate rows found!

4. COLUMN-BY-COLUMN DETAILED ANALYSIS

exercise_name:
  Type: object
  Missing: 0 (0.00%)
  Unique values: 1,506

category:
  Type: object
  Missing: 0 (0.00%)
  Unique values: 27

primary_muscles:
  Type: object
  Missing: 0 (0.00%)
  Unique values: 29

secondary_muscles:
  Type: object
  Missing: 0 (0.00%)
  Unique values: 245

equipment:
  Type: object
  Mis

# save data to drive


In [ ]:
df_combined.to_csv("/content/exercise_merged_normalized_data.csv")

file_source="/content/exercise_merged_normalized_data.csv"

dis_source="/content/drive/MyDrive/SPORT_METADATA/Structure data/Gym_exercises/"

shutil.copy(file_source, dis_source)


'/content/drive/MyDrive/SPORT_METADATA/Structure data/Gym_exercises/exercise_merged_normalized_data.csv'